In [ ]:
EXP_NAME="3JUN"
N = 20000
n_test = 2000
n_train = 2000
seed = 4
minority_perc = 0.4
# prevalence = 0.10

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import skewnorm, gamma, beta, truncnorm
from scipy.special import expit
from scipy.optimize import bisect
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import mutual_info_regression
from sklearn.utils import resample
import matplotlib.pyplot as plt
import seaborn as sns
from notebooks.analysis_utils import plot_cat_feature, plot_cont_feature
from data_utils import class_balanced_sampling, scale_dataset, apply_treatment_bias, apply_measurement_calibration_bias, apply_acuity_dependent_censoring

np.random.seed(seed)

# Generate the clinical ground truth 

## Latent health and outcome

In [ ]:
def generate_latent_health(n_pop, S):
  """ 
  Generates the latent health state of the population, underlying cause of the outcome
  """

  # Health Latent driving both clinical manifestion and target outcome
  health_latent = gamma.rvs(a=2, scale=1.5, size=n_pop)

  # Udesc, latent independent of the sensitive attribute
  u_desc = health_latent + gamma.rvs(a=2.1, scale=1, size=n_pop)

  # Ucorr, latent influenced by S
  u_corr = health_latent + np.where(S == 0, 0.3, 1.7) * gamma.rvs(a=1.2, scale=1.0, size=n_pop)

  return health_latent, u_corr, u_desc

In [ ]:
S = np.random.binomial(1, 1 - minority_perc, N)

health_latent, latent_corr, latent_desc = generate_latent_health(N, S)

Y_prob_raw = 2*latent_corr*latent_desc / (latent_corr + latent_desc) 
Y_prob = (Y_prob_raw - Y_prob_raw.min()) / (Y_prob_raw.max() - Y_prob_raw.min())

fig, axes = plt.subplots(4, 1, figsize=(16, 16))
sns.histplot(x=health_latent, common_norm=False, stat="probability", hue=S, ax=axes[0])
sns.histplot(x=latent_corr, common_norm=False, hue=S, stat="probability", ax=axes[1])
sns.histplot(x=latent_desc, common_norm=False, hue=S, stat="probability", ax=axes[2])
sns.histplot(x=Y_prob, common_norm=False, hue=S, stat="probability", ax=axes[3])
axes[0].set_title("health latent")
axes[1].set_title("latent 1 (corr)")
axes[2].set_title("latent 2 (desc)")
axes[3].set_title("Outcome probability")
plt.show()

In [ ]:
Y = (Y_prob > 0.3).astype(int)

prevalence = round(Y.sum() / len(Y) * 100, 2)
print(f"Outcome prevalence: {prevalence}%")

sns.histplot(x=Y, hue=S, common_norm=False, multiple='dodge', discrete=True, stat='probability')
plt.show()

## Clinical features

In [ ]:
def generate_clinical_ground_truth(n_pop, S, latent_corr, latent_desc):

  # --- Xcorr PATHWAY ---
  # Biomarker 1, 3: different means per S group
  biomarker_1 = 40 + 15*S + 5*latent_corr + np.random.normal(0, 10, n_pop)
  biomarker_3 = 80 - 40 * S + 8 * latent_corr + np.random.normal(0, 6, n_pop)

  # --- Xdesc PATHWAY ---
  # Biomarker 2, 4, 6: independent of S 
  biomarker_2 = np.random.normal(100 + 8 * latent_desc, 15, n_pop)

  # Biomarker 4: Logarithmic dependency (saturating effect)
  biomarker_4 = np.random.normal(50 + 15 * np.log(latent_desc + 1), 8, n_pop)
  
  # Biomarker 6: Log-Normal distribution
  mu_log_6 = np.log(latent_desc + 1)
  biomarker_6 = np.random.lognormal(mean=mu_log_6, sigma=0.25, size=n_pop)

  return pd.DataFrame({
    'S': S,
    'biomarker_1' : biomarker_1,
    'biomarker_2' : biomarker_2,
    'biomarker_3' : biomarker_3,
    'biomarker_4' : biomarker_4,
    'biomarker_6' : biomarker_6
  })

unbiased_pop = generate_clinical_ground_truth(N, S, latent_corr, latent_desc)
unbiased_pop['Y'] = Y

import os
data_dir = f"{PROJECT_ROOT}/data/synth/{EXP_NAME}"
os.makedirs(data_dir, exist_ok=True)

unbiased_pop.to_csv(f'{data_dir}/multivariate_raw.csv', index=False)

corr_columns = ['biomarker_1','biomarker_3']
raw_desc_columns = ['biomarker_2','biomarker_4','biomarker_6']


In [ ]:
from tableone import TableOne


# Descriptive statistics
table1 = TableOne(unbiased_pop,
                  groupby='S',
                  continuous= corr_columns + raw_desc_columns,
                  categorical=['Y'],
                  missing=False,
                  sort=True
                  )

print("Population statistics summary grouped by S")
print("="*80)
print(table1.tabulate(tablefmt = "fancy_grid"))

# Unbiased dataset

## Sampling

In [ ]:
# --- Unbiased sampling ---
# S-balanced sampling
unbiased_train, unbiased_test = train_test_split(
    unbiased_pop, 
    train_size=n_train, 
    test_size=n_test, 
    stratify=unbiased_pop['S'], 
    random_state=seed
)

# --- Class-balanced sampling ---
unbiased_bal_train, unbiased_bal_test = class_balanced_sampling(
    unbiased_pop,
    class_label = 'Y',
    n_train = n_train,
    n_test = n_test,
    seed = seed
)

In [ ]:
print("=== Unbiased dataset, S-stratified random sampling ===")
print(f"\nTarget Train size: {n_train} | Actual size: {len(unbiased_train)}")
print(f"Prevalence in Training set: {unbiased_train['Y'].mean():.2%}")
print(f"\nTarget Test size: {n_test} | Actual size: {len(unbiased_test)}")
print(f"Prevalence in Test set: {unbiased_test['Y'].mean():.2%}")

print("\n=== Unbiased dataset, class-balanced sampling ===")
print(f"\nTarget Train size: {n_train} | Actual size: {len(unbiased_bal_train)}")
print(f"Prevalence in Training set: {unbiased_bal_train['Y'].mean():.2%}")
print(f"\nTarget Test size: {n_test} | Actual size: {len(unbiased_bal_test)}")
print(f"Prevalence in Test set: {unbiased_bal_test['Y'].mean():.2%}")

In [ ]:
table1 = TableOne(unbiased_train,
                  groupby='S',
                  continuous= corr_columns + raw_desc_columns,
                  categorical=['Y'],
                  missing=False,
                  sort=True
                  )

print("FAIR DATASET, S-STRATIFIED RANDOM SAMPLING")
print("Statistics summary grouped by S")
print("="*80)
print(table1.tabulate(tablefmt = "fancy_grid"))

In [ ]:
table1 = TableOne(unbiased_bal_train,
                  groupby='S',
                  continuous= corr_columns + raw_desc_columns,
                  categorical=['Y'],
                  missing=False,
                  sort=True
                  )

print("FAIR DATASET, CLASS-BALANCED SAMPLING")
print("Statistics summary grouped by S")
print("="*80)
print(table1.tabulate(tablefmt = "fancy_grid"))

## Distributions

### Xcorr: S-correlated

In [ ]:
# Biomarker 1, influenced by S
print("=== Proportion-preserving sampling ===")
fig = plot_cont_feature(unbiased_test, 'biomarker_1', "Biomarker 1", "Y", 'S')
print("=== Class-balanced sampling ===")
fig = plot_cont_feature(unbiased_bal_test, 'biomarker_1', "Biomarker 1", "Y", 'S')

In [ ]:
# Biomarker 3, influenced by S
print("=== Proportion-preserving sampling ===")
fig = plot_cont_feature(unbiased_test, 'biomarker_3', "Biomarker 3", "Y", 'S')
print("=== Class-balanced sampling ===")
fig = plot_cont_feature(unbiased_bal_test, 'biomarker_3', "Biomarker 3", "Y", 'S')

### Xdesc: S-independent (biologically)

In [ ]:
# Biomarker 2, independent from S
print("=== Proportion-preserving sampling ===")
fig = plot_cont_feature(unbiased_test, 'biomarker_2', "Biomarker 2", "Y", 'S')
print("=== Class-balanced sampling ===")
fig = plot_cont_feature(unbiased_bal_test, 'biomarker_2', "Biomarker 2", "Y", 'S')

In [ ]:
# Biomarker 4, independent from S
print("=== Proportion-preserving sampling ===")
fig = plot_cont_feature(unbiased_test, 'biomarker_4', "Biomarker 4", "Y", 'S')
print("=== Class-balanced sampling ===")
fig = plot_cont_feature(unbiased_bal_test, 'biomarker_4', "Biomarker 4", "Y", 'S')

In [ ]:
# Biomarker 6, independent from S
print("=== Proportion-preserving sampling ===")
fig = plot_cont_feature(unbiased_test, 'biomarker_6', "Biomarker 6", "Y", 'S')
print("=== Class-balanced sampling ===")
fig = plot_cont_feature(unbiased_bal_test, 'biomarker_6', "Biomarker 6", "Y", 'S')

## Mutual Information with S

In [ ]:
def mutual_info_with_sens(X, sens):
  N_MI = 100
  mi_results = []

  for i in range(N_MI):
    X_resampled, sens_resampled = resample(X, sens, n_samples=100, random_state=i)
    mi_scores = mutual_info_regression(X_resampled, sens_resampled, n_neighbors=5, random_state=seed)
    mi_results.append(mi_scores)

  mi_df = pd.DataFrame(mi_results, columns=X.columns)
  mi_mean = mi_df.mean().sort_values(ascending=False)
  mi_df = mi_df[mi_mean.index]

  print("\n--- Mutual Information with S ---\n")
  print(mi_df.describe().to_markdown())

  plt.figure(figsize=(6, 3))
  ax = sns.barplot(data=mi_df, orient='h', estimator='median', palette='plasma')
  plt.title('Mutual Information Scores with Sensitive Attribute (100 bootstrap samples)')
  plt.show()

In [ ]:
# CONTROL
shuffled_S = unbiased_train['S'].sample(frac=1, random_state=42).values
mutual_info_with_sens(
  X=unbiased_train[ corr_columns + raw_desc_columns],
  sens=shuffled_S
)

In [ ]:
mutual_info_with_sens(
  X=unbiased_train[ corr_columns + raw_desc_columns],
  sens=unbiased_train['S']
)

In [ ]:
# Biased Dataset, class-balanced sampling
mutual_info_with_sens(
  X=unbiased_bal_train[ corr_columns + raw_desc_columns],
  sens=unbiased_bal_train['S']
)

# Biased datasets

## Apply bias to Desc path

In [ ]:
biased_df = apply_treatment_bias(
  unbiased_pop, "biomarker_2", 
  s_target=0, 
  bias_prob=1.0, 
  b_mean_shift=-30, 
  b_std_shift=10, 
  seed=seed
)

biased_df = apply_measurement_calibration_bias(
  df=biased_df, 
  biomarker="biomarker_4", 
  s_target=0, 
  bias_prob=1.0,         
  added_noise_std=15.0,  
  non_negative=False,   
  seed=seed
)

biased_df = apply_acuity_dependent_censoring(
  df=biased_df,
  biomarker="biomarker_6",
  s_target=0,
  bias_prob=1.0,          
  threshold_quantile=0.60, 
  attenuation_factor=0.15, 
  seed=seed
)

biased_desc_columns = ['biomarker_2_obs', 'biomarker_4_obs', 'biomarker_6_obs']



In [ ]:

fig = plot_cont_feature(biased_df, "biomarker_2_obs", "Observed Biomarker 2", "Y", 'S')


In [ ]:
fig = plot_cont_feature(biased_df, "biomarker_4_obs", "Observed Biomarker 4", "Y", 'S')

In [ ]:
fig = plot_cont_feature(biased_df, "biomarker_6_obs", "Observed Biomarker 6", "Y", 'S')

## Sampling

In [ ]:
# --- Unbiased sampling ---
# Random Sampling
biased_train, biased_test = train_test_split(
    biased_df, 
    train_size=n_train, 
    test_size=n_test, 
    stratify=biased_df['S'], 
    random_state=seed
)

# --- Class-balanced sampling ---
biased_classbal_train, biased_classbal_test = class_balanced_sampling(
    biased_df,
    class_label = 'Y',
    n_train = n_train,
    n_test = n_test,
    seed = seed
)

In [ ]:
print("=== Unbiased dataset, proportion-preserving sampling ===")
print(f"Target Train size: {n_train} | Actual size: {len(biased_train)}")
print(f"Prevalence in Training set: {biased_train['Y'].mean():.2%}")
print(f"Target Test size: {n_test} | Actual size: {len(biased_test)}")
print(f"Prevalence in Test set: {biased_test['Y'].mean():.2%}")

print("=== Unbiased dataset, class-balanced sampling ===")
print(f"Target Train size: {n_train} | Actual size: {len(biased_classbal_train)}")
print(f"Prevalence in Training set: {biased_classbal_train['Y'].mean():.2%}")
print(f"Target Test size: {n_test} | Actual size: {len(biased_classbal_test)}")
print(f"Prevalence in Test set: {biased_classbal_test['Y'].mean():.2%}")

## Summary stats

In [ ]:
table1 = TableOne(biased_train,
                  groupby='S',
                  continuous=corr_columns + biased_desc_columns,
                  categorical=['Y'],
                  missing=False,
                  sort=True
                  )

print("BIASED DATASET, S-STRATIFIED RANDOM SAMPLING")
print("Statistics summary grouped by S")
print("="*80)
print(table1.tabulate(tablefmt = "fancy_grid"))

In [ ]:
table1 = TableOne(biased_classbal_train,
                  groupby='S',
                  continuous=corr_columns + biased_desc_columns,
                  categorical=['Y'],
                  missing=False,
                  sort=True
                  )

print("BIASED DATASET, CLASS-BALANCED SAMPLING")
print("Statistics summary grouped by S")
print("="*80)
print(table1.tabulate(tablefmt = "fancy_grid"))

## Test Distributions

In [ ]:
print(f'======= PROPORTION PRESERVING SAMPLING =======')
fig = plot_cont_feature(biased_test, "biomarker_1", "Biomarker 1", "Y", 'S')
fig = plot_cont_feature(biased_test, "biomarker_3", "Biomarker 3", "Y", 'S')
fig = plot_cont_feature(biased_test, "biomarker_2_obs", "Observed Biomarker 2", "Y", 'S')
fig = plot_cont_feature(biased_test, "biomarker_4_obs", "Observed Biomarker 4", "Y", 'S')
fig = plot_cont_feature(biased_test, "biomarker_6_obs", "Observed Biomarker 6", "Y", 'S')
print(f'======= CLASS-BALANCED SAMPLING =======')
fig = plot_cont_feature(biased_classbal_test, "biomarker_1", "Biomarker 1", "Y", 'S')
fig = plot_cont_feature(biased_classbal_test, "biomarker_3", "Biomarker 3", "Y", 'S')
fig = plot_cont_feature(biased_classbal_test, "biomarker_2_obs", "Observed Biomarker 2", "Y", 'S')
fig = plot_cont_feature(biased_classbal_test, "biomarker_4_obs", "Observed Biomarker 4", "Y", 'S')
fig = plot_cont_feature(biased_classbal_test, "biomarker_6_obs", "Observed Biomarker 6", "Y", 'S')

## Mutual Information with S

In [ ]:
# Biased Dataset, random sampling
mutual_info_with_sens(
  X=biased_train[corr_columns + raw_desc_columns + biased_desc_columns],
  sens=biased_train['S']
)

In [ ]:
# Biased Dataset, class-balanced sampling
mutual_info_with_sens(
  X=biased_classbal_train[corr_columns + raw_desc_columns + biased_desc_columns],
  sens=biased_classbal_train['S']
)

# Scaling

In [ ]:
minmax_features = corr_columns
standard_features = ['biomarker_2', 'biomarker_4']
skewed_features = ['biomarker_6']

unbiased_train_processed, unbiased_test_processed = scale_dataset(
  unbiased_train, unbiased_test, 
  minmax_features=minmax_features,
  standard_features=standard_features,
  skewed_features=skewed_features)

minmax_features = corr_columns
standard_features = ['biomarker_2', 'biomarker_4']
skewed_features = ['biomarker_6']

unbiased_classbal_train_processed, unbiased_classbal_test_processed = scale_dataset(
  unbiased_bal_train, unbiased_bal_test, 
  minmax_features=minmax_features,
  standard_features=standard_features,
  skewed_features=skewed_features)

unbiased_train_processed.to_csv(f'{data_dir}/multivariate_fair_test.csv', index=False)
unbiased_test_processed.to_csv(f'{data_dir}/multivariate_fair_train.csv', index=False)
unbiased_classbal_train_processed.to_csv(f'{data_dir}/multivariate_fair_classbal_train.csv', index=False)
unbiased_classbal_test_processed.to_csv(f'{data_dir}/multivariate_fair_classbal_test.csv', index=False)

In [ ]:
minmax_features = corr_columns + biased_desc_columns
standard_features = []
skewed_features = []

biased_train_processed, biased_test_processed = scale_dataset(
  biased_train, biased_test, 
  minmax_features=minmax_features,
  standard_features=standard_features,
  skewed_features=skewed_features)
biased_classbal_test_processed, biased_classbal_train_processed = scale_dataset(
  biased_classbal_train, biased_classbal_test, 
  minmax_features=minmax_features,
  standard_features=standard_features,
  skewed_features=skewed_features)

biased_train_processed.to_csv(f'{data_dir}/multivariate_biased_test.csv', index=False)
biased_test_processed.to_csv(f'{data_dir}/multivariate_biased_train.csv', index=False)
biased_classbal_train_processed.to_csv(f'{data_dir}/multivariate_biased_classbal_train.csv', index=False)
biased_classbal_test_processed.to_csv(f'{data_dir}/multivariate_biased_classbal_test.csv', index=False)